In [76]:
! pip install accelerator

In [77]:
import pandas as pd
import numpy as np

In [78]:
# load the dataset and check the file
df = pd.read_csv("/kaggle/input/dataset/spam.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [79]:
# check the count of spam and Non-spam
df.Category.value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [80]:
# new column for spam = 1 and ham = 0
df["spam"] = df["Category"].apply(lambda x: 1 if x == "spam" else 0)

In [81]:
df.head()

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [82]:
df.shape

(5572, 3)

In [83]:
from sklearn.model_selection import train_test_split

In [84]:
# creating train and test dataset
# train_test_split for spliting the dataset
x_train,x_test,y_train,y_test = train_test_split(df["Message"],df["spam"],test_size = 0.2)

In [85]:
x_train.shape

(4457,)

In [86]:
x_test.shape

(1115,)

In [87]:
x_train[:4]

259     We tried to contact you re your reply to our o...
907     all the lastest from Stereophonics, Marley, Di...
4947             I'm already back home so no probably not
492     Congrats! 1 year special cinema pass for 2 is ...
Name: Message, dtype: object

In [88]:
x_train[:4]

259     We tried to contact you re your reply to our o...
907     all the lastest from Stereophonics, Marley, Di...
4947             I'm already back home so no probably not
492     Congrats! 1 year special cinema pass for 2 is ...
Name: Message, dtype: object

In [89]:
type(y_train)

pandas.core.series.Series

In [90]:
y_train[:4]

259     1
907     1
4947    0
492     1
Name: spam, dtype: int64

In [91]:
# for sklearn countvectorizer
from sklearn.feature_extraction.text import CountVectorizer

In [92]:
# for BOW representation
v = CountVectorizer()
x_train_cv = v.fit_transform(x_train.values)
x_train_cv

<4457x7785 sparse matrix of type '<class 'numpy.int64'>'
	with 59360 stored elements in Compressed Sparse Row format>

In [93]:
# to view the Series
x_train_cv.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [94]:
# to check the shape
x_train_cv.shape

(4457, 7785)

In [95]:
x_test.shape

(1115,)

In [96]:
x_train_np = x_train_cv.toarray()
x_train_np

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [97]:
np.where(x_train_np[1] != 0)

(array([ 660,  914,  965, 1428, 1838, 2374, 2897, 3021, 3069, 4033, 4106,
        4365, 4814, 4980, 5557, 6495, 6546, 6808, 6835, 6847, 6962, 7419,
        7560, 7586]),)

In [98]:
# Model building using naive bayes
from sklearn.naive_bayes import MultinomialNB

In [99]:
model = MultinomialNB()
model.fit(x_train_cv, y_train)

MultinomialNB()

In [100]:
# creating test dataset and evaluating
x_test_cv = v.transform(x_test)

In [101]:
# evaluate the model
from sklearn.metrics import classification_report

In [103]:
# prediction
y_predict = model.predict(x_test_cv)
print(classification_report(y_test,y_predict))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       961
           1       0.99      0.90      0.95       154

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [104]:
# some check
# 0 for ham and 1 for spam
emails = [
    "Hey Mohan, can we together for a football match tonight?",
    "Upto 20% discount on parking, exclusive offer just for you. Don't miss this reward!"
]
emails_cv = v.transform(emails)
model.predict(emails_cv)

array([0, 1])

In [106]:
# Easy method via pipeline
from sklearn.pipeline import Pipeline

In [108]:
# creating the pipeline
clf = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("nb", MultinomialNB())
])

In [109]:
clf.fit(x_train, y_train)

Pipeline(steps=[('vectorizer', CountVectorizer()), ('nb', MultinomialNB())])

In [111]:
y_predict = classification_report(y_test,clf.predict(x_test))
print(y_predict)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       961
           1       0.99      0.90      0.95       154

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

